# Portfolio Efficient Frontier v2 — Live Market Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frankhuettner/ManSciDA-Excel-to-app/blob/main/portfolio-app-v2/portfolio_v2.ipynb)

**Model:** Quadratic Program (QP)

$$\min_{w} \quad w^\top \hat{\Sigma}\, w$$
$$\text{s.t.} \quad \sum_i w_i = 1, \quad \hat{r}^\top w \geq r^*, \quad w \geq 0 \;\text{(optional)}$$

where $\hat{r}$ and $\hat{\Sigma}$ are estimated from **historical daily log returns**, annualised with 252 trading days.

**Solver:** CVXPY + Clarabel  
**Data:** Yahoo Finance via `yfinance`  
**Same solver code** runs in the browser app (`portfolio-app-v2/index.html`) via Pyodide — only the data source differs.

---

| Context | Runtime | Data source |
|---|---|---|
| This notebook | Google Colab / local Python (`uv sync`) | `yfinance` (direct) |
| Browser app | Pyodide (WebAssembly) | Yahoo Finance via corsproxy.io |

In [ ]:
# Install packages — runs automatically in Google Colab; skip locally if already installed
import sys
if 'google.colab' in sys.modules:
    %pip install yfinance cvxpy clarabel -q

In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

print(f'cvxpy    {cp.__version__}')
print(f'numpy    {np.__version__}')
print(f'yfinance {yf.__version__}')

## User Configuration

Edit the variables below, then **Run All** (or re-run from here) to rebuild the frontier with new tickers or settings.

In [ ]:
TICKERS  = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']  # any Yahoo Finance ticker symbols
PERIOD   = '2y'    # lookback: '1y', '2y', '3y', '5y'
RF_RATE  = 0.045   # annualised risk-free rate (e.g. 0.045 = 4.5 %)
NO_SHORT = True    # True = weights must be >= 0 (no short selling)
N_PTS    = 60      # number of target-return steps along the frontier

## Download Historical Prices

`yfinance` fetches adjusted closing prices directly from Yahoo Finance.

In [ ]:
raw = yf.download(TICKERS, period=PERIOD, auto_adjust=True, progress=False)

# yfinance >= 0.2 returns MultiIndex columns; handle both single and multi-ticker
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close'][TICKERS]
else:
    prices = raw[['Close']].rename(columns={'Close': TICKERS[0]})

prices = prices.dropna()
print(f'Tickers : {list(prices.columns)}')
print(f'Period  : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Trading days with complete data: {len(prices)}')
prices.tail(3)

## Annualised Returns and Covariance

Daily log returns $r_t = \ln(P_t / P_{t-1})$ are annualised by multiplying by 252 trading days.

In [ ]:
log_ret = np.log(prices / prices.shift(1)).dropna()

ann_ret = log_ret.mean() * 252          # annualised expected returns
ann_cov = log_ret.cov() * 252           # annualised covariance matrix
ann_std = np.sqrt(np.diag(ann_cov.values))

summary = pd.DataFrame({
    'Ann. Return': ann_ret.map('{:.2%}'.format),
    'Ann. Std Dev': pd.Series(ann_std, index=ann_ret.index).map('{:.2%}'.format),
    'Sharpe (rf={:.1%})'.format(RF_RATE): (
        (ann_ret - RF_RATE) / pd.Series(ann_std, index=ann_ret.index)
    ).map('{:.3f}'.format),
})
print('Individual asset statistics:')
display(summary)
print('\nCorrelation matrix (daily log returns):')
display(log_ret.corr().round(3))

## QP Solver

The solver function is **identical** to the browser app — only the inputs differ (historical data here vs. a manual CORS-proxied fetch in the browser).

In [ ]:
r     = ann_ret.values          # 1-D numpy array of annualised returns
S     = ann_cov.values          # covariance matrix
n     = len(r)
rf    = RF_RATE
NAMES = list(prices.columns)

def solve_qp(r_target, no_short=NO_SHORT):
    """Minimise portfolio variance subject to a minimum target return."""
    w    = cp.Variable(n)
    cons = [cp.sum(w) == 1, r @ w >= r_target]
    if no_short:
        cons.append(w >= 0)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, S)), cons)
    prob.solve(solver=cp.CLARABEL, verbose=False)
    if prob.status not in ('optimal', 'optimal_inaccurate') or w.value is None:
        return None
    wv  = w.value
    ret = float(r @ wv)
    std = float(np.sqrt(max(float(wv @ S @ wv), 0.0)))
    return wv, ret, std

## Efficient Frontier

Sweep `r_target` from just above the risk-free rate to 130 % of the highest individual asset return.

In [ ]:
r_lo = rf + 0.001
r_hi = float(np.max(r)) * 1.3
if r_hi <= r_lo:
    r_hi = r_lo + 0.15

targets = np.linspace(r_lo, r_hi, N_PTS)

rows = []
for rt in targets:
    res = solve_qp(rt)
    if res is not None:
        wv, ret, std = res
        shp = (ret - rf) / std if std > 1e-9 else 0.0
        rows.append({
            'return':  ret,
            'std':     std,
            'sharpe':  shp,
            **{f'w_{nm}': wv[i] for i, nm in enumerate(NAMES)},
        })

frontier = pd.DataFrame(rows)
print(f'Feasible frontier points: {len(frontier)} / {N_PTS}')
frontier[['return', 'std', 'sharpe']].mul([100, 100, 1]).round(3).head(8)

In [ ]:
i_mv = frontier['std'].idxmin()
i_ms = frontier['sharpe'].idxmax()

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(frontier['std'] * 100, frontier['return'] * 100,
        color='#7c3aed', lw=2, label='Efficient frontier')

for i, nm in enumerate(NAMES):
    ax.scatter(ann_std[i] * 100, r[i] * 100,
               s=90, marker='^', color='#f59e0b',
               label='Assets' if i == 0 else None, zorder=5)
    ax.annotate(nm, (ann_std[i] * 100 + 0.25, r[i] * 100), fontsize=9)

ax.scatter(frontier.loc[i_mv, 'std'] * 100, frontier.loc[i_mv, 'return'] * 100,
           s=130, marker='D', color='#10b981', zorder=6, label='Min variance')
ax.scatter(frontier.loc[i_ms, 'std'] * 100, frontier.loc[i_ms, 'return'] * 100,
           s=180, marker='*', color='#3b82f6', zorder=6, label='Max Sharpe')

ax.set_xlabel('Standard deviation (%)', fontsize=11)
ax.set_ylabel('Expected return (%)', fontsize=11)
ax.set_title(
    f'Efficient Frontier — {", ".join(NAMES)}  ({PERIOD}, rf={RF_RATE:.1%})',
    fontsize=12
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Portfolios

In [ ]:
key = frontier.loc[[i_mv, i_ms]].copy()
key.index = ['Min Variance', 'Max Sharpe']

display_cols = ['return', 'std', 'sharpe'] + [f'w_{nm}' for nm in NAMES]
key[display_cols].mul([100, 100, 1] + [1] * n).round(3).rename(
    columns={'return': 'Return %', 'std': 'Std %', 'sharpe': 'Sharpe'}
)